# PC-MMIA Feature Extraction

Extracts PC-MMIA features (Zheng et al., EMNLP Findings 2025) for train, val and test splits.

**Resumable:** If the runtime disconnects, re-run from Cell 4 onwards — it will automatically pick up from the last checkpoint.

**Output:** Saves CSVs to `milestone6/data/extracted_features/4_pcmmia/`

Created with support from Claude AI. Comments are my own. 

In [ ]:
# 1: Install packages
!pip install -q transformers datasets torch pillow tqdm
!pip install -q num2words

In [ ]:
# 2: Mount Drive and set paths
from google.colab import drive
drive.mount('/content/drive')

import sys
import os

# Path where milestone6 folder exists on Drive
MILESTONE_DIR = '/content/drive/MyDrive/milestone6'
SRC_DIR       = f'{MILESTONE_DIR}/src'
OUTPUT_DIR    = f'{MILESTONE_DIR}/data/extracted_features/4_pcmmia'

os.makedirs(OUTPUT_DIR, exist_ok=True)
sys.path.insert(0, SRC_DIR)

print(f'Source dir:  {SRC_DIR}')
print(f'Output dir:  {OUTPUT_DIR}')
print(f'Contents of src: {os.listdir(SRC_DIR)}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Source dir:  /content/drive/MyDrive/milestone6/src
Output dir:  /content/drive/MyDrive/milestone6/data/extracted_features/4_pcmmia
Contents of src: ['lightweight_pipeline_method_3.py', 'pcmmia_extract.ipynb', 'pcmmia_features.py']


In [ ]:
# 3: Load model and dataset 
# Only run once in a session, skip if model already loaded

import torch
from transformers import AutoProcessor, SmolVLMForConditionalGeneration
from datasets import load_dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dtype  = torch.bfloat16 if device.type == 'cuda' and torch.cuda.is_bf16_supported() else torch.float32
print(f'Device: {device}  |  dtype: {dtype}')

MODEL_ID   = 'UBC-SLIME/colx_585_vlm'
DATASET_ID = 'UBC-SLIME/colx585_group_project_data'

print('Loading processor...')
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
processor.image_processor.do_image_splitting = False
processor.image_processor.size = {'longest_edge': 512}
processor.image_processor.max_image_size = {'longest_edge': 512}

print('Loading model...')
model = SmolVLMForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    _attn_implementation='sdpa',
    trust_remote_code=True,
).to(device).eval()
print('Model loaded.')

print('Loading dataset...')
dataset = load_dataset(DATASET_ID)
splits = {
    'train':      dataset['train'].to_pandas(),
    'validation': dataset['validation'].to_pandas(),
    'test':       dataset['test'].to_pandas(),
}
print(f"Train: {len(splits['train'])}  Val: {len(splits['validation'])}  Test: {len(splits['test'])}")

Device: cuda  |  dtype: torch.bfloat16
Loading processor...
Loading model...


model.safetensors:   0%|          | 0.00/513M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/471 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/135 [00:00<?, ?B/s]

Model loaded.
Loading dataset...


...

test.parquet:   0%|          | 0.00/338M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1200 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6000 [00:00<?, ? examples/s]

Train: 6000  Val: 1200  Test: 6000


In [ ]:
# 4: Extract training set features
# If runtime disconnects, re-run the cell. Extraction will re-commence from last saved checkpoint.

from pcmmia_features import extract_pcmmia_for_split

extract_pcmmia_for_split(
    model        = model,
    processor    = processor,
    df           = splits['train'],
    save_path    = f'{OUTPUT_DIR}/train.csv',
    has_labels   = True,
    checkpoint_every = 100,
)

Starting fresh — 6000 samples to process



PC-MMIA extraction:   2%|▏         | 103/6000 [00:05<04:59, 19.69it/s]


Checkpoint saved — 100 / 6000 samples done



PC-MMIA extraction:   3%|▎         | 202/6000 [00:10<04:56, 19.54it/s]


Checkpoint saved — 200 / 6000 samples done


...


Checkpoint saved — 5800 / 6000 samples done



PC-MMIA extraction:  98%|█████████▊| 5902/6000 [05:01<00:04, 20.11it/s]


Checkpoint saved — 5900 / 6000 samples done



PC-MMIA extraction: 100%|██████████| 6000/6000 [05:06<00:00, 19.57it/s]


Checkpoint saved — 6000 / 6000 samples done
Done. CSV saved to /content/drive/MyDrive/milestone6/data/extracted_features/4_pcmmia/train.csv


In [ ]:
# 5: Extract validation set features
# If runtime disconnects, re-run the cell. Extraction will re-commence from last saved checkpoint.

from pcmmia_features import extract_pcmmia_for_split

extract_pcmmia_for_split(
    model        = model,
    processor    = processor,
    df           = splits['validation'],
    save_path    = f'{OUTPUT_DIR}/val.csv',
    has_labels   = True,
    checkpoint_every = 100,
)

Starting fresh — 1200 samples to process


PC-MMIA extraction:   8%|▊         | 102/1200 [00:05<00:56, 19.37it/s]


Checkpoint saved — 100 / 1200 samples done


PC-MMIA extraction:  17%|█▋        | 203/1200 [00:10<00:50, 19.71it/s]


Checkpoint saved — 200 / 1200 samples done


...


Checkpoint saved — 1000 / 1200 samples done


PC-MMIA extraction:  92%|█████████▏| 1102/1200 [00:56<00:04, 19.88it/s]


Checkpoint saved — 1100 / 1200 samples done


PC-MMIA extraction: 100%|██████████| 1200/1200 [01:02<00:00, 19.33it/s]


Checkpoint saved — 1200 / 1200 samples done
Done. CSV saved to /content/drive/MyDrive/milestone6/data/extracted_features/4_pcmmia/val.csv


In [ ]:
# 6: Extract test set features
# If runtime disconnects, re-run the cell. Extraction will re-commence from last saved checkpoint.

from pcmmia_features import extract_pcmmia_for_split

extract_pcmmia_for_split(
    model        = model,
    processor    = processor,
    df           = splits['test'],
    save_path    = f'{OUTPUT_DIR}/test.csv',
    has_labels   = False,
    checkpoint_every = 100,
)

Starting fresh — 6000 samples to process


PC-MMIA extraction:   2%|▏         | 103/6000 [00:05<05:00, 19.64it/s]


Checkpoint saved — 100 / 6000 samples done


PC-MMIA extraction:   3%|▎         | 204/6000 [00:10<04:51, 19.90it/s]


Checkpoint saved — 200 / 6000 samples done


...


Checkpoint saved — 5800 / 6000 samples done


PC-MMIA extraction:  98%|█████████▊| 5902/6000 [04:59<00:04, 20.15it/s]


Checkpoint saved — 5900 / 6000 samples done


PC-MMIA extraction: 100%|██████████| 6000/6000 [05:04<00:00, 19.70it/s]


Checkpoint saved — 6000 / 6000 samples done
Done. CSV saved to /content/drive/MyDrive/milestone6/data/extracted_features/4_pcmmia/test.csv


In [ ]:
# 7: Check output 

import pandas as pd

for split in ['train', 'val', 'test']:
    path = f'{OUTPUT_DIR}/{split}.csv'
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f'{split}: {len(df)} rows  |  columns: {list(df.columns)}')
        print(df.head(2))
        print()
    else:
        print(f'{split}: NOT FOUND')

train: 6000 rows  |  columns: ['pcmmia_score', 'pcmmia_mean_delta', 'pcmmia_max_delta', 'pcmmia_std_delta', 'pcmmia_top10_delta', 'pcmmia_top25_delta', 'id', 'label']
   pcmmia_score  pcmmia_mean_delta  pcmmia_max_delta  pcmmia_std_delta  \
0     -1.826850           0.000779           0.12500           0.04275   
1     -2.195138          -0.004379           0.09375           0.04693   

   pcmmia_top10_delta  pcmmia_top25_delta                 id  label  
0            0.092773            0.057292  cauldron_ln_15810      0  
1            0.071289            0.049006  cauldron_ln_24846      0  

val: 1200 rows  |  columns: ['pcmmia_score', 'pcmmia_mean_delta', 'pcmmia_max_delta', 'pcmmia_std_delta', 'pcmmia_top10_delta', 'pcmmia_top25_delta', 'id', 'label']
   pcmmia_score  pcmmia_mean_delta  pcmmia_max_delta  pcmmia_std_delta  \
0     -2.253561           0.002032          0.187500          0.049297   
1     -1.154198          -0.003193          0.074219          0.031827   

   pcmmia_t